# 家庭关系本体

此示例是 [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) 的一部分，灵感来源于[这篇博客文章](https://habr.com/post/270857/)。

我总是觉得记住家庭成员之间的各种关系很困难。在这个示例中，我们将使用一个定义家庭关系的本体，以及实际的家谱树，展示如何通过自动推理来找到所有的亲属关系。

### 获取家谱树

作为示例，我们将使用[罗曼诺夫沙皇家族](https://en.wikipedia.org/wiki/House_of_Romanov)的家谱。描述家庭关系最常见的格式是 [GEDCOM](https://en.wikipedia.org/wiki/GEDCOM)。我们将使用 GEDCOM 格式的罗曼诺夫家族树：


In [21]:
!head -15 /data/tsars.ged

'head' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


要使用GEDCOM文件，我们可以使用`python-gedcom`库：


In [3]:
import sys
!{sys.executable} -m pip install python-gedcom

这个库解决了一些文件解析的技术问题，但它仍然为我们提供了对树中所有个人和家庭的相当低级的访问。以下是我们如何解析文件并显示所有个人列表的方法：


In [30]:
from gedcom.parser import Parser
from gedcom.element.individual import IndividualElement
from gedcom.element.family import FamilyElement
g = Parser()
g.parse_file('data/tsars.ged')

FileNotFoundError: [Errno 2] No such file or directory: 'data/tsars.ged'

In [22]:
d = g.get_element_dictionary()
[ (k,v.get_name()) for k,v in d.items() if isinstance(v,IndividualElement)]

[]

以下是我们获取家庭信息的方法。请注意，这会给我们一个**标识符**列表，如果我们想要更清楚，需要将它们转换为名称：


In [23]:
d = g.get_element_dictionary()
[ (k,[x.get_value() for x in v.get_child_elements()]) for k,v in d.items() if isinstance(v,FamilyElement)]

[]

### 获取家庭本体

接下来，让我们看看[家庭本体](https://raw.githubusercontent.com/blokhin/genealogical-trees/master/data/header.ttl)，它被定义为一组语义网三元组。这个本体定义了诸如 `isUncleOf`、`isCousinOf` 等许多关系。所有这些关系都是基于基本谓词 `isMotherOf`、`isFatherOf`、`isBrotherOf` 和 `isSisterOf` 定义的。我们将使用自动推理，通过本体推导出所有其他关系。

以下是 `isAuntOf` 属性的一个示例定义，它被定义为 `isSisterOf` 和 `isParentOf` 的组合（*姑/姨是某人父母的姐妹*）。A是B的姐妹，B是C的父母

```
fhkb:isAuntOf a owl:ObjectProperty ;
    rdfs:domain fhkb:Woman ;
    rdfs:range fhkb:Person ;
    owl:propertyChainAxiom ( fhkb:isSisterOf fhkb:isParentOf ) .
```


In [25]:
!head -20 data/onto.ttl

'head' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


### 构建推理本体

为了简化操作，我们将创建一个本体文件，其中包括家庭本体中的原始规则，以及来自我们 GEDCOM 文件的个人事实。我们将遍历 GEDCOM 文件，提取有关家庭和个人的信息，并将其转换为三元组。


In [26]:
!cp data/onto.ttl .

gedcom_dict = g.get_element_dictionary()
individuals, marriages = {}, {}

def term2id(el):
    return "i" + el.get_pointer().replace('@', '').lower()

out = open("onto.ttl","a")

for k, v in gedcom_dict.items():
    if isinstance(v,IndividualElement):
        children, siblings = set(), set()
        idx = term2id(v)

        title = v.get_name()[0] + " " + v.get_name()[1]
        title = title.replace('"', '').replace('[', '').replace(']', '').replace('(', '').replace(')', '').strip()

        own_families = g.get_families(v, 'FAMS')
        for fam in own_families:
            children |= set(term2id(i) for i in g.get_family_members(fam, "CHIL"))

        parent_families = g.get_families(v, 'FAMC')
        if len(parent_families):
            for member in g.get_family_members(parent_families[0], "CHIL"): # NB adoptive families i.e len(parent_families)>1 are not considered (TODO?)
                if member.get_pointer() == v.get_pointer():
                    continue
                siblings.add(term2id(member))

        if idx in individuals:
            children |= individuals[idx].get('children', set())
            siblings |= individuals[idx].get('siblings', set())
        individuals[idx] = {'sex': v.get_gender().lower(), 'children': children, 'siblings': siblings, 'title': title}

    elif isinstance(v,FamilyElement):
        wife, husb, children = None, None, set()
        children = set(term2id(i) for i in g.get_family_members(v, "CHIL"))

        try:
            wife = g.get_family_members(v, "WIFE")[0]
            wife = term2id(wife)
            if wife in individuals: individuals[wife]['children'] |= children
            else: individuals[wife] = {'children': children}
        except IndexError: pass
        try:
            husb = g.get_family_members(v, "HUSB")[0]
            husb = term2id(husb)
            if husb in individuals: individuals[husb]['children'] |= children
            else: individuals[husb] = {'children': children}
        except IndexError: pass

        if wife and husb: marriages[wife + husb] = (term2id(v), wife, husb)

for idx, val in individuals.items():
    added_terms = ''
    if val['sex'] == 'f':
        parent_predicate, sibl_predicate = "isMotherOf", "isSisterOf"
    else:
        parent_predicate, sibl_predicate = "isFatherOf", "isBrotherOf"
    if len(val['children']):
        added_terms += " ;\n    fhkb:" + parent_predicate + " " + ", ".join(["fhkb:" + i for i in val['children']])
    if len(val['siblings']):
        added_terms += " ;\n    fhkb:" + sibl_predicate + " " + ", ".join(["fhkb:" + i for i in val['siblings']])
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing%s ;\n    rdfs:label \"%s\" .\n" % (idx, added_terms, val['title']))

for k, v in marriages.items():
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing ;\n    fhkb:hasFemalePartner fhkb:%s ;\n    fhkb:hasMalePartner fhkb:%s .\n" % v)

out.write("[] a owl:AllDifferent ;\n    owl:distinctMembers (")
for idx in individuals.keys():
    out.write("    fhkb:" + idx)
for k, v in marriages.items():
    out.write("    fhkb:" + v[0])
out.write("    ) .")
out.close()

'cp' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [28]:
!tail onto.ttl

'tail' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


### 推理操作

现在我们希望能够使用这个本体进行推理和查询。我们将使用 [RDFLib](https://github.com/RDFLib)，一个用于读取不同格式的 RDF 图、查询等操作的库。

对于逻辑推理，我们将使用 [OWL-RL](https://github.com/RDFLib/OWL-RL) 库，它允许我们构建 RDF 图的**闭包**，即添加所有可以推导出的概念和关系。


In [29]:
!{sys.executable} -m pip install rdflib
!{sys.executable} -m pip install git+https://github.com/RDFLib/OWL-RL.git

   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ------------------ --------------------- 262.1/569.0 kB ? eta -:--:--
   ---------------------------------------- 569.0/569.0 kB 1.0 MB/s  0:00:00
  Cloning https://github.com/RDFLib/OWL-RL.git to c:\users\huawei\appdata\local\temp\pip-req-build-xxh_lt7n
  Resolved https://github.com/RDFLib/OWL-RL.git to commit 8e1bdbb322c656fcb7f6745447d740eec40fc18e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for owlrl: fi

  Running command git clone --filter=blob:none --quiet https://github.com/RDFLib/OWL-RL.git 'C:\Users\huawei\AppData\Local\Temp\pip-req-build-xxh_lt7n'


让我们打开本体文件，看看它包含多少个三元组：


In [11]:
import rdflib
from owlrl import DeductiveClosure, OWLRL_Extension

g = rdflib.Graph()
g.parse("onto.ttl", format="turtle")

print("Triplets found:%d" % len(g))

Triplets found:669


现在让我们构建闭包，看看三元组的数量如何增加：


In [12]:
DeductiveClosure(OWLRL_Extension).expand(g)
print("Triplets after inference:%d" % len(g))

Triplets after inference:4246


### 查询亲属关系

现在我们可以查询图谱，查看人与人之间的不同关系。我们可以结合使用 **SPARQL** 语言和 `query` 方法。在我们的例子中，让我们看看家谱中所有的**叔叔**：


In [13]:
qres = g.query(
    """SELECT DISTINCT ?aname ?bname
       WHERE {
          ?a fhkb:isUncleOf ?b .
          ?a rdfs:label ?aname .
          ?b rdfs:label ?bname .
       }""")

for row in qres:
    print("%s is uncle of %s" % row)

Fedor Alekseevich Romanov is uncle of Ekaterina Ivanovna Romanova
Aleksandr I Pavlovich Romanov is uncle of Aleksandr II Nikolaevich Romanov
Fedor Alekseevich Romanov is uncle of Anna Ivanovna Romanova


可以尝试不同的家庭关系。例如，可以查看 `isAncestorOf` 关系，它递归地定义了某个人的所有祖先。

最后，让我们整理一下！


In [14]:
!rm onto.ttl


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。虽然我们尽力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
